# Continuous Authentication

Primary task: classify each session as **Authorized** or **Unauthorized**.

The notebook uses a participant-aware split so that sessions from one participant do not appear in both training and test data.

## 1. Load Dataset

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import make_scorer
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix, f1_score,
    precision_score, recall_score, roc_auc_score
)
from sklearn.model_selection import GroupKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import GroupShuffleSplit

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')

DATA_PATHS = [
    Path(r'A:\AI\ML\Data\continuous_auth_dataset.csv'),
    Path.cwd() / 'continuous_auth_dataset.csv',
]
DATA_PATH = next((path for path in DATA_PATHS if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError('continuous_auth_dataset.csv was not found in the expected locations.')

df = pd.read_csv(DATA_PATH)
print(f'Loaded dataset from: {DATA_PATH}')
print(f'Dataset shape: {df.shape}')


## 2. Data Understanding

In [ ]:
display(df.head())
print('Data types:')
display(df.dtypes.to_frame('dtype'))
print('Missing values:', int(df.isna().sum().sum()))
print('Exact duplicate rows:', int(df.duplicated().sum()))
print('Unique participants:', df['Participant_ID'].nunique())
print('Unique sessions:', df['Session_ID'].nunique())
print('Target distribution:')
display(df['True_Class'].value_counts().to_frame('count'))
display((df['True_Class'].value_counts(normalize=True) * 100).round(2).to_frame('percent'))


## 3. Data Cleaning

In [ ]:
df['Timestamp'] = pd.to_datetime(df['Timestamp'], errors='raise')

# The dataset has no missing or duplicate rows, but keep these checks explicit.
assert df.isna().sum().sum() == 0, 'Missing values require review before modeling.'
assert df.duplicated().sum() == 0, 'Duplicate rows require review before modeling.'

range_checks = {
    'Typing_Accuracy_pct': (0, 100),
    'CPU_Usage_pct': (0, 100),
    'Memory_Usage_pct': (0, 100),
    'Previous_Authentication_Score': (0, 1),
}
for column, (lower, upper) in range_checks.items():
    assert df[column].between(lower, upper).all(), f'Invalid values found in {column}.'

print('Cleaning checks passed.')


## 4. Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='True_Class', hue='True_Class', legend=False, palette='Set2')
plt.title('Target Distribution')
plt.tight_layout()
plt.show()

numeric_columns = [
    'Keyboard_Speed_WPM', 'Keystroke_Duration_ms', 'Typing_Accuracy_pct',
    'Mouse_Speed_pxs', 'CPU_Usage_pct', 'Memory_Usage_pct',
    'Previous_Authentication_Score'
]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for axis, column in zip(axes.flat, numeric_columns):
    sns.boxplot(data=df, x='True_Class', y=column, hue='True_Class', legend=False, ax=axis)
    axis.set_title(column)
for axis in axes.flat[len(numeric_columns):]:
    axis.axis('off')
plt.tight_layout()
plt.show()

print('Authentication fields compared with the true class:')
display(pd.crosstab(df['Authentication_Label'], df['True_Class']))


## 5. Define Target and Remove Leakage Columns

The target is `True_Class`. The authentication score and decision fields are removed because they may be generated after, or directly from, the authentication decision. Using them without clear documentation would make the evaluation misleading.

In [ ]:
TARGET = 'True_Class'
LEAKAGE_COLUMNS = [
    'Authentication_Score', 'Authentication_Label',
    'Access_Action', 'Security_Level'
]

# These columns are identifiers or raw text, not model inputs in this project.
IDENTIFIER_COLUMNS = ['Session_ID', 'Timestamp', 'Language_Text']
GROUP_COLUMN = 'Participant_ID'

X = df.drop(columns=[TARGET, *LEAKAGE_COLUMNS, *IDENTIFIER_COLUMNS, GROUP_COLUMN])
y = df[TARGET].copy()
groups = df[GROUP_COLUMN].copy()

print('Features used for classification:')
print(X.columns.tolist())
print('Target classes:', y.unique().tolist())


## 6. Participant-Aware Train/Test Split

In [ ]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_indices, test_indices = next(splitter.split(X, y, groups=groups))

X_train = X.iloc[train_indices].copy()
X_test = X.iloc[test_indices].copy()
y_train = y.iloc[train_indices].copy()
y_test = y.iloc[test_indices].copy()
groups_train = groups.iloc[train_indices].copy()
groups_test = groups.iloc[test_indices].copy()

overlapping_participants = set(groups_train).intersection(set(groups_test))
assert len(overlapping_participants) == 0

print('Training rows:', len(X_train))
print('Test rows:', len(X_test))
print('Training participants:', groups_train.nunique())
print('Test participants:', groups_test.nunique())
print('Overlapping participants:', len(overlapping_participants))
print('Training target distribution:')
display(y_train.value_counts(normalize=True).round(3).to_frame('proportion'))
print('Test target distribution:')
display(y_test.value_counts(normalize=True).round(3).to_frame('proportion'))


## 7. Simple Feature Engineering

In [ ]:
def add_features(features, timestamps):
    result = features.copy()
    timestamp_values = pd.to_datetime(timestamps)
    result['Hour'] = timestamp_values.dt.hour.to_numpy()
    result['Is_Weekend'] = (timestamp_values.dt.dayofweek >= 5).astype(int).to_numpy()
    result['Hour_Sin'] = np.sin(2 * np.pi * result['Hour'] / 24)
    result['Hour_Cos'] = np.cos(2 * np.pi * result['Hour'] / 24)
    speed = result['Keyboard_Speed_WPM'].clip(lower=1e-6)
    result['Keystroke_Efficiency'] = result['Keystroke_Duration_ms'] / speed
    result['Mouse_to_Keyboard_Ratio'] = result['Mouse_Speed_pxs'] / speed
    result['Typing_Accuracy_Adjusted_Speed'] = (
        result['Keyboard_Speed_WPM'] * result['Typing_Accuracy_pct'] / 100
    )
    return result

X_train = add_features(X_train, df.iloc[train_indices]['Timestamp'])
X_test = add_features(X_test, df.iloc[test_indices]['Timestamp'])

print('Final numeric and categorical features are prepared.')


## 8. Preprocessing Pipeline

In [ ]:
numeric_features = X_train.select_dtypes(include='number').columns.tolist()
categorical_features = X_train.select_dtypes(include='object').columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', StandardScaler(), numeric_features),
        ('categorical', OneHotEncoder(handle_unknown='ignore'), categorical_features),
    ]
)


## 9. Model Training

In [ ]:
models = {
    'Dummy Baseline': DummyClassifier(strategy='most_frequent'),
    'Logistic Regression': LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42),
    'Random Forest': RandomForestClassifier(
        n_estimators=50, class_weight='balanced', random_state=42, n_jobs=1
    ),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
}

cv = GroupKFold(n_splits=3)
model_rows = []
fitted_models = {}

for model_name, model in models.items():
    pipeline = Pipeline([('preprocessor', preprocessor), ('model', model)])
    scores = cross_validate(
        pipeline, X_train, y_train, groups=groups_train, cv=cv,
        scoring={
        'accuracy': 'accuracy',
        'precision': make_scorer(precision_score, pos_label='Unauthorized', zero_division=0),
        'recall': make_scorer(recall_score, pos_label='Unauthorized', zero_division=0),
        'f1': make_scorer(f1_score, pos_label='Unauthorized', zero_division=0),
        'roc_auc': 'roc_auc',
    },
        n_jobs=1
    )
    model_rows.append({
        'Model': model_name,
        'CV Accuracy': scores['test_accuracy'].mean(),
        'CV Precision': scores['test_precision'].mean(),
        'CV Recall': scores['test_recall'].mean(),
        'CV F1': scores['test_f1'].mean(),
        'CV ROC-AUC': scores['test_roc_auc'].mean(),
    })
    fitted_models[model_name] = pipeline.fit(X_train, y_train)

model_comparison = pd.DataFrame(model_rows).sort_values(
    ['CV F1', 'CV ROC-AUC', 'CV Accuracy'], ascending=False
).reset_index(drop=True)
display(model_comparison.round(4))


## 10. Model Evaluation

In [ ]:
evaluation_rows = []

for model_name, pipeline in fitted_models.items():
    train_predictions = pipeline.predict(X_train)
    test_predictions = pipeline.predict(X_test)
    test_probabilities = pipeline.predict_proba(X_test)[:, 1] if hasattr(pipeline, 'predict_proba') else None

    evaluation_rows.append({
        'Model': model_name,
        'Training Accuracy': accuracy_score(y_train, train_predictions),
        'Test Accuracy': accuracy_score(y_test, test_predictions),
        'Test Precision': precision_score(y_test, test_predictions, pos_label='Unauthorized', zero_division=0),
        'Test Recall': recall_score(y_test, test_predictions, pos_label='Unauthorized', zero_division=0),
        'Test F1': f1_score(y_test, test_predictions, pos_label='Unauthorized', zero_division=0),
        'Test ROC-AUC': roc_auc_score((y_test == 'Unauthorized').astype(int), test_probabilities)
        if test_probabilities is not None else np.nan,
    })

evaluation_results = pd.DataFrame(evaluation_rows)
evaluation_results['CV order'] = evaluation_results['Model'].map(
    {name: position for position, name in enumerate(model_comparison['Model'])}
)
evaluation_results = evaluation_results.sort_values('CV order').drop(columns='CV order').reset_index(drop=True)
display(evaluation_results.round(4))

# Select the final model using grouped cross-validation, before looking at test results.
best_model_name = model_comparison.iloc[0]['Model']
best_model = fitted_models[best_model_name]
best_predictions = best_model.predict(X_test)
best_probabilities = best_model.predict_proba(X_test)[:, 1]

print('Selected model:', best_model_name)
print('Classification report:')
print(classification_report(y_test, best_predictions, zero_division=0))

confusion = confusion_matrix(y_test, best_predictions, labels=['Authorized', 'Unauthorized'])
sns.heatmap(
    confusion, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Authorized', 'Unauthorized'],
    yticklabels=['Authorized', 'Unauthorized']
)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f'Confusion Matrix - {best_model_name}')
plt.tight_layout()
plt.show()


## 11. Final Model Selection and Conclusion

In [ ]:
train_accuracy = accuracy_score(y_train, best_model.predict(X_train))
test_accuracy = accuracy_score(y_test, best_predictions)
accuracy_gap = train_accuracy - test_accuracy

print(f'Final model: {best_model_name}')
print(f'Training accuracy: {train_accuracy:.4f}')
print(f'Test accuracy: {test_accuracy:.4f}')
print(f'Test F1-score: {f1_score(y_test, best_predictions, pos_label="Unauthorized", zero_division=0):.4f}')
print(f'Test ROC-AUC: {roc_auc_score((y_test == "Unauthorized").astype(int), best_probabilities):.4f}')

if accuracy_gap > 0.10:
    interpretation = 'The model shows signs of overfitting because training accuracy is much higher than test accuracy.'
elif test_accuracy < 0.60:
    interpretation = 'The model may be underfitting because test accuracy is low.'
else:
    interpretation = 'The training and test results are reasonably close, so the model appears reasonably balanced.'
print(interpretation)
print('The final model was selected using grouped cross-validation, with F1-score considered first because both authentication classes matter. The test set was used only for the final report.')


### Additional Analyses

Regression, anomaly detection, and clustering were removed from the main workflow because classification of `True_Class` is the primary project objective. They can be added later as separate, clearly labeled experiments after the classification pipeline is validated.

## Feature Ablation Experiment

This is a simple comparison using the final Random Forest pipeline. Experiment A uses all current features. Experiment B removes only `Active_Application` and `Command_Type`; the participant-aware split, preprocessing, grouped cross-validation, model, and metrics remain the same.


In [ ]:
from sklearn.metrics import make_scorer

def run_ablation_experiment(name, dropped_features):
    selected_features = [column for column in X_train.columns if column not in dropped_features]
    numeric_columns = X_train[selected_features].select_dtypes(include='number').columns.tolist()
    categorical_columns = X_train[selected_features].select_dtypes(include='object').columns.tolist()

    ablation_preprocessor = ColumnTransformer(
        transformers=[
            ('numeric', StandardScaler(), numeric_columns),
            ('categorical', OneHotEncoder(handle_unknown='ignore'), categorical_columns),
        ]
    )

    ablation_pipeline = Pipeline([
        ('preprocessor', ablation_preprocessor),
        ('model', RandomForestClassifier(
            n_estimators=50, class_weight='balanced', random_state=42, n_jobs=1
        )),
    ])

    ablation_scoring = {
        'f1': make_scorer(f1_score, pos_label='Unauthorized', zero_division=0),
        'roc_auc': 'roc_auc',
    }
    cv_scores = cross_validate(
        ablation_pipeline,
        X_train[selected_features],
        y_train,
        groups=groups_train,
        cv=GroupKFold(n_splits=3),
        scoring=ablation_scoring,
        n_jobs=1,
    )

    ablation_pipeline.fit(X_train[selected_features], y_train)
    train_predictions = ablation_pipeline.predict(X_train[selected_features])
    test_predictions = ablation_pipeline.predict(X_test[selected_features])
    test_probabilities = ablation_pipeline.predict_proba(X_test[selected_features])[:, 1]

    return {
        'Experiment': name,
        'CV F1': cv_scores['test_f1'].mean(),
        'CV ROC-AUC': cv_scores['test_roc_auc'].mean(),
        'Train Accuracy': accuracy_score(y_train, train_predictions),
        'Test Accuracy': accuracy_score(y_test, test_predictions),
        'Test Precision': precision_score(y_test, test_predictions, pos_label='Unauthorized', zero_division=0),
        'Test Recall': recall_score(y_test, test_predictions, pos_label='Unauthorized', zero_division=0),
        'Test F1': f1_score(y_test, test_predictions, pos_label='Unauthorized', zero_division=0),
        'Test ROC-AUC': roc_auc_score((y_test == 'Unauthorized').astype(int), test_probabilities),
    }

ablation_results = pd.DataFrame([
    run_ablation_experiment('A. Full Features', []),
    run_ablation_experiment(
        'B. Without Application and Command Features',
        ['Active_Application', 'Command_Type'],
    ),
])
display(ablation_results.round(4))

full_results = ablation_results.iloc[0]
reduced_results = ablation_results.iloc[1]
print('Change in Test Accuracy:', round(full_results['Test Accuracy'] - reduced_results['Test Accuracy'], 4))
print('Change in Test F1:', round(full_results['Test F1'] - reduced_results['Test F1'], 4))
print('Change in Test ROC-AUC:', round(full_results['Test ROC-AUC'] - reduced_results['Test ROC-AUC'], 4))


## Feature Ablation Conclusion

Removing `Active_Application` and `Command_Type` causes a large performance decrease. The reduced model still learns useful information from behavioral, system, network, and historical features, but it is clearly weaker than the full model. The full model should remain the main project model because the experiment does not prove that these features are leakage; it proves that they contain major predictive signal in this dataset. Both results should be shown in the final report as a transparent feature ablation comparison, and the dataset's rule-like design should be acknowledged.


## Reduced Behavior Model Tuning

The reduced model excludes `Active_Application` and `Command_Type`. A small manual Random Forest search is used to reduce unnecessary complexity. The participant-aware split and grouped cross-validation remain unchanged.


In [ ]:
reduced_features = [
    column for column in X_train.columns
    if column not in ['Active_Application', 'Command_Type']
]

def evaluate_reduced_random_forest(model_name, **model_parameters):
    numeric_columns = X_train[reduced_features].select_dtypes(include='number').columns.tolist()
    categorical_columns = X_train[reduced_features].select_dtypes(include='object').columns.tolist()
    reduced_preprocessor = ColumnTransformer(
        transformers=[
            ('numeric', StandardScaler(), numeric_columns),
            ('categorical', OneHotEncoder(handle_unknown='ignore'), categorical_columns),
        ]
    )
    reduced_pipeline = Pipeline([
        ('preprocessor', reduced_preprocessor),
        ('model', RandomForestClassifier(
            n_estimators=50, class_weight='balanced', random_state=42, n_jobs=1,
            **model_parameters
        )),
    ])
    tuning_scoring = {
        'f1': make_scorer(f1_score, pos_label='Unauthorized', zero_division=0),
        'roc_auc': 'roc_auc',
    }
    cv_scores = cross_validate(
        reduced_pipeline, X_train[reduced_features], y_train,
        groups=groups_train, cv=GroupKFold(n_splits=3),
        scoring=tuning_scoring, n_jobs=1,
    )
    reduced_pipeline.fit(X_train[reduced_features], y_train)
    train_predictions = reduced_pipeline.predict(X_train[reduced_features])
    test_predictions = reduced_pipeline.predict(X_test[reduced_features])
    test_probabilities = reduced_pipeline.predict_proba(X_test[reduced_features])[:, 1]
    train_accuracy = accuracy_score(y_train, train_predictions)
    test_accuracy = accuracy_score(y_test, test_predictions)
    return {
        'Model': model_name,
        'CV F1': cv_scores['test_f1'].mean(),
        'CV ROC-AUC': cv_scores['test_roc_auc'].mean(),
        'Train Accuracy': train_accuracy,
        'Test Accuracy': test_accuracy,
        'Test Precision': precision_score(y_test, test_predictions, pos_label='Unauthorized', zero_division=0),
        'Test Recall': recall_score(y_test, test_predictions, pos_label='Unauthorized', zero_division=0),
        'Test F1': f1_score(y_test, test_predictions, pos_label='Unauthorized', zero_division=0),
        'Test ROC-AUC': roc_auc_score((y_test == 'Unauthorized').astype(int), test_probabilities),
        'Train-Test Accuracy Gap': train_accuracy - test_accuracy,
        'max_depth': model_parameters.get('max_depth'),
        'min_samples_split': model_parameters.get('min_samples_split', 2),
        'min_samples_leaf': model_parameters.get('min_samples_leaf', 1),
        'max_features': model_parameters.get('max_features', 'default'),
    }

tuned_reduced_results = pd.DataFrame([
    evaluate_reduced_random_forest('Current reduced RF'),
    evaluate_reduced_random_forest(
        'Tuned RF depth 8', max_depth=8, min_samples_split=5,
        min_samples_leaf=2, max_features='sqrt'
    ),
    evaluate_reduced_random_forest(
        'Tuned RF depth 12', max_depth=12, min_samples_split=5,
        min_samples_leaf=2, max_features='sqrt'
    ),
    evaluate_reduced_random_forest(
        'Tuned RF depth 8 leaf 4', max_depth=8, min_samples_split=10,
        min_samples_leaf=4, max_features='sqrt'
    ),
    evaluate_reduced_random_forest(
        'Tuned RF depth 12 leaf 2', max_depth=12, min_samples_split=10,
        min_samples_leaf=2, max_features='sqrt'
    ),
])
tuned_reduced_results = tuned_reduced_results.sort_values(
    ['CV F1', 'CV ROC-AUC'], ascending=False
).reset_index(drop=True)
display(tuned_reduced_results.round(4))

selected_reduced_model = tuned_reduced_results.iloc[0]
print('Selected reduced model:', selected_reduced_model['Model'])
print('Selected hyperparameters:')
print({key: selected_reduced_model[key] for key in [
    'max_depth', 'min_samples_split', 'min_samples_leaf', 'max_features'
]})
baseline_gap = tuned_reduced_results.loc[
    tuned_reduced_results['Model'] == 'Current reduced RF', 'Train-Test Accuracy Gap'
].iloc[0]
selected_gap = selected_reduced_model['Train-Test Accuracy Gap']
print('Train-test accuracy gap before tuning:', round(baseline_gap, 4))
print('Train-test accuracy gap after tuning:', round(selected_gap, 4))
print('Gap reduction:', round(baseline_gap - selected_gap, 4))


### Reduced Model Tuning Conclusion

The selected reduced model uses `max_depth=12`, `min_samples_split=10`, `min_samples_leaf=2`, and `max_features='sqrt'`. Its test F1-score improves from 0.8406 to 0.8511, while the train-test accuracy gap decreases from 0.1369 to 0.0924. Overfitting is reduced but not completely removed. The tuned reduced model is a better behavior-focused alternative, while the full-feature Random Forest remains the main project model.


In [ ]:
# Final project model: behavior-focused Random Forest.
final_model_parameters = {
    'max_depth': 12,
    'min_samples_split': 10,
    'min_samples_leaf': 2,
    'max_features': 'sqrt',
}

final_numeric_columns = X_train[reduced_features].select_dtypes(include='number').columns.tolist()
final_categorical_columns = X_train[reduced_features].select_dtypes(include='object').columns.tolist()
final_preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', StandardScaler(), final_numeric_columns),
        ('categorical', OneHotEncoder(handle_unknown='ignore'), final_categorical_columns),
    ]
)
final_project_pipeline = Pipeline([
    ('preprocessor', final_preprocessor),
    ('model', RandomForestClassifier(
        n_estimators=50,
        class_weight='balanced',
        random_state=42,
        n_jobs=1,
        **final_model_parameters,
    )),
])
final_project_pipeline.fit(X_train[reduced_features], y_train)
final_train_predictions = final_project_pipeline.predict(X_train[reduced_features])
final_test_predictions = final_project_pipeline.predict(X_test[reduced_features])
final_test_probabilities = final_project_pipeline.predict_proba(X_test[reduced_features])[:, 1]

print('Final project model: Tuned Behavior-Focused Random Forest')
print('Excluded features:', ['Active_Application', 'Command_Type'])
print('Final test accuracy:', round(accuracy_score(y_test, final_test_predictions), 4))
print('Final test F1-score:', round(f1_score(y_test, final_test_predictions, pos_label='Unauthorized'), 4))
print('Final test ROC-AUC:', round(roc_auc_score((y_test == 'Unauthorized').astype(int), final_test_probabilities), 4))


## Final Project Model Decision

The project adopts the **Tuned Behavior-Focused Random Forest** as the final model. It excludes `Active_Application` and `Command_Type` and uses `max_depth=12`, `min_samples_split=10`, `min_samples_leaf=2`, and `max_features='sqrt'`.

The full-feature Random Forest remains a comparison model. The final behavior-focused results are approximately 86.65% test accuracy, 85.11% test F1-score, and 94.59% test ROC-AUC.


In [ ]:
import joblib
from pathlib import Path

model_path = Path('tuned_behavior_focused_pipeline.pkl')
joblib.dump(final_project_pipeline, model_path)

print(f'Model saved to: {model_path.resolve()}')

# Load the saved pipeline when predictions are needed.
loaded_pipeline = joblib.load(model_path)
predictions = loaded_pipeline.predict(X_test[reduced_features])
probabilities = loaded_pipeline.predict_proba(X_test[reduced_features])[:, 1]

print('Saved pipeline loaded successfully.')
print('Example predictions:', predictions[:5])
print('Example probabilities:', probabilities[:5])
